<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Fine_Tuning_Qwen3_on_Your_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*More details in this article: [Fine-Tuning Qwen3 or Qwen3 Base?](https://kaitchup.substack.com/p/fine-tuning-qwen3-or-qwen3-base)*


This notebook fine-tunes Qwen3 models with Unsloth. It shows different configurations, fine-tuning base and post-trained models, with a chat template.

The second part demonstrates how to use the fine-tuned adapters and their impact at inference time.

A GPU with 24 GB of VRAM is enough to run this notebook. If you only have a 16 GB GPU, use Qwen3-8B models instead.

# Installation

In [ ]:
%%capture
#Code by Unsloth
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
!pip install flash_attn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 48.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 98.5 MB/s eta 0:00:00
  Created wheel for flash_attn: filename=flash_attn-2.7.4.post1-cp311-cp311-lin

# Fine-Tuning Qwen3 Base Models

In [ ]:
from unsloth import FastLanguageModel
import torch, os, multiprocessing, gc
from datasets import load_dataset
from peft import LoraConfig
from transformers import set_seed, AutoTokenizer

from trl import SFTTrainer, SFTConfig

set_seed(42)

model_name = "Qwen/Qwen3-14B-Base"
compute_dtype = torch.bfloat16

bs = 1 #Batch size per device (training and validation), bs = 1 *can* be faster
gas = 64 #Gradient accumulation steps
mseqlen = 2048 #Maximum sequence length to 1024 for a 24 GB GPU or use Qwen3-8B; reduce if you run out of memory

lr = 1e-4

output_dir = "./SFT-Base/"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = mseqlen,
    dtype = compute_dtype,
    load_in_4bit=True
)

ds_train = load_dataset("allenai/tulu-3-sft-mixture", split="train[:150000]")

#Bug? Unsloth seems not loading the chat template for the base model, even though it is in the tokenizer_config.json
tokenizer.chat_template = AutoTokenizer.from_pretrained("Qwen/Qwen3-14B-Base").chat_template
def process(row):
    row["text"] = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False)
    return row

ds_train = ds_train.map(
    process,
    num_proc= multiprocessing.cpu_count(),
    load_from_cache_file=False,
)

print(ds_train[0]['text'])

ds_train = ds_train.remove_columns(["messages"])

model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj","embed_tokens", "lm_head"],
    lora_alpha = 32,
    use_gradient_checkpointing = 'unsloth', # True or "unsloth" for very long context
    random_state = 3407,
)
from unsloth import UnslothTrainer, UnslothTrainingArguments

training_arguments = UnslothTrainingArguments(
        output_dir=output_dir,
        #eval_strategy="steps",
        #do_eval=True,
        optim="paged_adamw_8bit",
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=gas,
        #per_device_eval_batch_size=bs,
        log_level="debug",
        save_strategy="steps",
        save_steps=250,
        logging_steps=25,
        learning_rate = lr,
        embedding_learning_rate = lr/10,
        bf16 = True,
        #eval_steps=25,
        max_steps=1000,
        warmup_ratio=0.03,
        report_to = "none",
        lr_scheduler_type="linear",
        max_seq_length=mseqlen,
        dataset_text_field='text',
        dataset_num_proc=multiprocessing.cpu_count()
)

trainer = UnslothTrainer(
    model = model,
    train_dataset=ds_train,
    #eval_dataset=ds_test,
    processing_class=tokenizer,
    args = training_arguments
)


#--code by Unsloth: https://colab.research.google.com/drive/1Ys44kVvmeZtnICzWz0xgpRnrIOjZAuxp?usp=sharing#scrollTo=pCqnaKmlO1U9

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

trainer_ = trainer.train()


used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_trainer= round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
trainer_percentage = round(used_memory_for_trainer/max_memory*100, 3)
print(f"{trainer_.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_trainer} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {trainer_percentage} %.")
print("-----")
#----

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.4.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/169k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.19G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

...

Step,Training Loss
25,1.543700
50,1.269400
75,1.289600
100,1.234700
125,1.265800
150,1.208300
175,1.264300
200,1.228700
225,1.214800
250,1.239400


Saving model checkpoint to ./drive/MyDrive/Qwen3/SFT-Base/checkpoint-250
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--unsloth--qwen3-14b-base-unsloth-bnb-4bit/snapshots/ff6c963a46e9634290a5c53dc8adbc3f47ceba95/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 5120,
  "initializer_range": 0.02,
  "intermediate_size": 17408,
  "max_position_embeddings": 32768,
  "max_window_layers": 40,
  "model_type": "qwen3",
  "num_attention_heads": 40,
  "num_hidden_layers": 40,
  "num_key_value_heads": 8,
  "pad_token_id": 151654,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_ena

Step,Training Loss
25,1.543700
50,1.269400
75,1.289600
100,1.234700
125,1.265800
150,1.208300
175,1.264300
200,1.228700
225,1.214800
250,1.239400


Saving model checkpoint to ./drive/MyDrive/Qwen3/SFT-Base/checkpoint-1000
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--unsloth--qwen3-14b-base-unsloth-bnb-4bit/snapshots/ff6c963a46e9634290a5c53dc8adbc3f47ceba95/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 5120,
  "initializer_range": 0.02,
  "intermediate_size": 17408,
  "max_position_embeddings": 32768,
  "max_window_layers": 40,
  "model_type": "qwen3",
  "num_attention_heads": 40,
  "num_hidden_layers": 40,
  "num_key_value_heads": 8,
  "pad_token_id": 151654,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_en

40757.8795 seconds used for training.
679.3 minutes used for training.
Peak reserved memory = 19.467 GB.
Peak reserved memory for training = 5.401 GB.
Peak reserved memory % of max memory = 49.213 %.
Peak reserved memory for training % of max memory = 13.654 %.
-----


# Fine-Tuning Qwen3 Reasoning Models

## Without Retraining the Embeddings

In [ ]:
from unsloth import FastLanguageModel
import torch, os, multiprocessing, gc
from datasets import load_dataset
from peft import LoraConfig
from transformers import set_seed, AutoTokenizer,DataCollatorForSeq2Seq

from trl import SFTTrainer, SFTConfig

set_seed(42)

model_name = "Qwen/Qwen3-14B"
compute_dtype = torch.bfloat16

bs = 1 #Batch size per device (training and validation), bs = 1 *can* be faster
gas = 64 #Gradient accumulation steps
mseqlen = 2048 #Maximum sequence length; reduce if you run out of memory

lr = 1e-4

output_dir = "./SFT/"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = mseqlen,
    dtype = compute_dtype,
    load_in_4bit=True
)

ds_train = load_dataset("allenai/tulu-3-sft-mixture", split="train[:150000]")

def process(row):
    row["text"] = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False)
    return row

ds_train = ds_train.map(
    process,
    num_proc= multiprocessing.cpu_count(),
    load_from_cache_file=False,
)

print(ds_train[0]['text'])

ds_train = ds_train.remove_columns(["messages"])

model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    use_gradient_checkpointing = 'unsloth', # True or "unsloth" for very long context
    random_state = 3407,
)
from unsloth import UnslothTrainer, UnslothTrainingArguments

training_arguments = UnslothTrainingArguments(
        output_dir=output_dir,
        #eval_strategy="steps",
        #do_eval=True,
        optim="paged_adamw_8bit",
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=gas,
        #per_device_eval_batch_size=bs,
        log_level="debug",
        save_strategy="steps",
        save_steps=250,
        logging_steps=25,
        learning_rate = lr,
        bf16 = True,
        #eval_steps=25,
        max_steps=1000,
        warmup_ratio=0.03,
        report_to = "none",
        lr_scheduler_type="linear",
        max_seq_length=mseqlen,
        dataset_text_field='text',
        dataset_num_proc=multiprocessing.cpu_count()
)

trainer = UnslothTrainer(
    model = model,
    train_dataset=ds_train,
    #eval_dataset=ds_test,
    processing_class=tokenizer,
    args = training_arguments
)


#--code by Unsloth: https://colab.research.google.com/drive/1Ys44kVvmeZtnICzWz0xgpRnrIOjZAuxp?usp=sharing#scrollTo=pCqnaKmlO1U9

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

trainer_ = trainer.train()


used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_trainer= round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
trainer_percentage = round(used_memory_for_trainer/max_memory*100, 3)
print(f"{trainer_.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_trainer} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {trainer_percentage} %.")
print("-----")
#----

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.4.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Map (num_proc=12):   0%|          | 0/150000 [00:00<?, ? examples/s]

<|im_start|>user
Create a snippet of Terraform HCL code that create an AWS autoscaling group, and an ALB in front to expose an application to internet.<|im_end|>
<|im_start|>assistant
<think>

</think>

Sure, here's an example Terraform HCL code that creates an AWS Autoscaling Group and an Application Load Balancer to expose an application to the internet:
``` 
# Configure the AWS provider
provider "aws" {
  region = "us-east-1"
}

# Create a security group to allow traffic to the ALB
resource "aws_security_group" "alb_sg" {
  name_prefix = "alb_sg"
  ingress {
    from_port = 80
    to_port = 80
    protocol = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }
}

# Create an ALB and target group
resource "aws_lb" "alb" {
  name               = "example-alb"
  internal           = false
  load_balancer_type = "application"

  subnets = ["subnet-12345678", "subnet-87654321"]

  security_groups = [aws_security_group.alb_sg.id]

  tags = {
    Environment = "production"
  }
}

resource "aws_lb_tar

Unsloth 2025.4.7 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.
max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend


GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
14.066 GB of memory reserved.


Currently training with a batch size of: 1
The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: id, text, source. If id, text, source are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 150,000 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 64
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 64 x 1) = 64
 "-____-"     Trainable parameters = 128,450,560/14,000,000,000 (0.92% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,1.918600


Step,Training Loss
25,1.918600


## With Retraining the Embeddings

In [ ]:
from unsloth import FastLanguageModel
import torch, os, multiprocessing, gc
from datasets import load_dataset
from peft import LoraConfig
from transformers import set_seed, AutoTokenizer,DataCollatorForSeq2Seq

from trl import SFTTrainer, SFTConfig

set_seed(42)

model_name = "Qwen/Qwen3-14B"
compute_dtype = torch.bfloat16

bs = 1 #Batch size per device (training and validation), bs = 1 *can* be faster
gas = 64 #Gradient accumulation steps
mseqlen = 2048 #Maximum sequence length; reduce if you run out of memory

lr = 1e-4

output_dir = "./SFT-emb/"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = mseqlen,
    dtype = compute_dtype,
    load_in_4bit=True
)

ds_train = load_dataset("allenai/tulu-3-sft-mixture", split="train[:150000]")

def process(row):
    row["text"] = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False)
    return row

ds_train = ds_train.map(
    process,
    num_proc= multiprocessing.cpu_count(),
    load_from_cache_file=False,
)

print(ds_train[0]['text'])

ds_train = ds_train.remove_columns(["messages"])

model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj" ,"embed_tokens", "lm_head"],
    lora_alpha = 32,
    use_gradient_checkpointing = 'unsloth', # True or "unsloth" for very long context
    random_state = 3407,
)
from unsloth import UnslothTrainer, UnslothTrainingArguments

training_arguments = UnslothTrainingArguments(
        output_dir=output_dir,
        #eval_strategy="steps",
        #do_eval=True,
        optim="paged_adamw_8bit",
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=gas,
        #per_device_eval_batch_size=bs,
        log_level="debug",
        save_strategy="steps",
        save_steps=250,
        logging_steps=25,
        learning_rate = lr,
        embedding_learning_rate = lr/10,
        bf16 = True,
        #eval_steps=25,
        max_steps=1000,
        warmup_ratio=0.03,
        report_to = "none",
        lr_scheduler_type="linear",
        max_seq_length=mseqlen,
        dataset_text_field='text',
        dataset_num_proc=multiprocessing.cpu_count()
)

trainer = UnslothTrainer(
    model = model,
    train_dataset=ds_train,
    #eval_dataset=ds_test,
    processing_class=tokenizer,
    args = training_arguments
)


#--code by Unsloth: https://colab.research.google.com/drive/1Ys44kVvmeZtnICzWz0xgpRnrIOjZAuxp?usp=sharing#scrollTo=pCqnaKmlO1U9

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

trainer_ = trainer.train()


used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_trainer= round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
trainer_percentage = round(used_memory_for_trainer/max_memory*100, 3)
print(f"{trainer_.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_trainer} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {trainer_percentage} %.")
print("-----")
#----

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.4.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/168k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.59G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

...

Currently training with a batch size of: 1
The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: id, text, source. If id, text, source are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 150,000 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 64
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 64 x 1) = 64
 "-____-"     Trainable parameters = 1,684,275,200/14,000,000,000 (12.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,1.897400
50,1.483000
75,1.440500
100,1.371900
125,1.389100
150,1.337100
175,1.372700
200,1.336000
225,1.320900
250,1.342700


Saving model checkpoint to ./drive/MyDrive/Qwen3/SFT-emb/checkpoint-250
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--unsloth--qwen3-14b-unsloth-bnb-4bit/snapshots/f359a260469e181f79110e57314815c09bf436af/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 5120,
  "initializer_range": 0.02,
  "intermediate_size": 17408,
  "max_position_embeddings": 40960,
  "max_window_layers": 40,
  "model_type": "qwen3",
  "num_attention_heads": 40,
  "num_hidden_layers": 40,
  "num_key_value_heads": 8,
  "pad_token_id": 151654,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp

Step,Training Loss
25,1.897400
50,1.483000
75,1.440500
100,1.371900
125,1.389100
150,1.337100
175,1.372700
200,1.336000
225,1.320900
250,1.342700


# Test the Fine-Tuned Adapters

In [ ]:
!pip install --upgrade transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

# Baseline (no Adapter)
## With Reasoning

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen3-14B"
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


# prepare the model input
prompt = "Let a, b be positive real numbers such that ab = a + b + 3. Determine the range of possible values for a + b."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

...

model-00002-of-00008.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00005-of-00008.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

thinking content: <think>
Okay, so I need to figure out the range of possible values for a + b given that a and b are positive real numbers satisfying ab = a + b + 3. Hmm, let me start by understanding the problem.

First, we have two variables, a and b, both positive real numbers. The equation given is ab = a + b + 3. And we need to find all possible values of a + b. So essentially, I need to find the minimum and maximum possible values of a + b under the given constraint. Wait, but since a and b are positive, maybe there's a minimum but no maximum? Or maybe both? Let me think.

Let me try to manipulate the equation to express it in terms of a + b. Let me denote S = a + b and P = ab. The given equation is P = S + 3. So, we have P = S + 3. Now, for two numbers a and b, there's a relationship between S and P. For example, in quadratic equations, if we know S and P, we can write the quadratic equation as x² - Sx + P = 0. But since a and b are real numbers, the discriminant of this quadra

## Without Reasoning

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen3-14B"
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


# prepare the model input
prompt = "Let a, b be positive real numbers such that ab = a + b + 3. Determine the range of possible values for a + b."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

thinking content: 
content: We are given the equation:

$$
ab = a + b + 3
$$

with $ a, b > 0 $, and we are to **determine the range of possible values** for $ a + b $.

---

### Step 1: Use substitution

Let:

$$
s = a + b \quad \text{and} \quad p = ab
$$

From the given equation:

$$
p = s + 3
$$

So we now have:

$$
ab = a + b + 3 \Rightarrow p = s + 3
$$

We are to find the **range of $ s = a + b $** given that $ ab = s + 3 $ and $ a, b > 0 $.

---

### Step 2: Use the relationship between $ a + b $ and $ ab $

We know from algebra that for any two real numbers $ a $ and $ b $, the quadratic equation with roots $ a $ and $ b $ is:

$$
x^2 - sx + p = 0
$$

In our case, $ p = s + 3 $, so the quadratic becomes:

$$
x^2 - sx + (s + 3) = 0
$$

We want this quadratic to have **positive real roots** $ a $ and $ b $.

For this to happen:

- The **discriminant** must be **non-negative** (i.e., real roots)
- The **roots must be positive** (since $ a, b > 0 $)

---

### Step 3: Compute the di

# With an Adapter Fine-Tuned for the Base Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
model_name = "unsloth/qwen3-14b-unsloth-bnb-4bit"
adapter_path = "./SFT-Base/checkpoint-750/"
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
mode = PeftModel.from_pretrained(model, adapter_path)


# prepare the model input
prompt = "Let a, b be positive real numbers such that ab = a + b + 3. Determine the range of possible values for a + b."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

thinking content: 
content:  szczegółowo

Okay, so I need to find the range of possible values for a + b given that a and b are positive real numbers satisfying ab = a + b + 3. Hmm, let's see. I remember that when dealing with equations involving products and sums of variables, sometimes substitution or using inequalities like AM ≥ GM can help. But let me try to work through this step by step.

First, let me write down the given equation:

ab = a + b + 3.

I need to find the range of a + b. Let me denote S = a + b and P = ab. Then, according to the problem, P = S + 3. But I also know from algebra that for two numbers, there's a relationship between their sum and product. Maybe I can express one variable in terms of the other or use substitution?

Alternatively, maybe I can rewrite the equation ab - a - b = 3. Let me try adding 1 to both sides:

ab - a - b + 1 = 4.

Hmm, that looks like it can be factored. Let me check:

(ab - a - b + 1) = (a - 1)(b - 1). Yes! Because (a - 1)(b - 1) = a

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
model_name = "unsloth/qwen3-14b-unsloth-bnb-4bit"
adapter_path = "./SFT-Base/checkpoint-750/"
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
mode = PeftModel.from_pretrained(model, adapter_path)


# prepare the model input
prompt = "Let a, b be positive real numbers such that ab = a + b + 3. Determine the range of possible values for a + b."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

thinking content: 
content: We are given that:

$$
ab = a + b + 3
$$

where $ a, b > 0 $. We are to determine the **range of possible values for $ a + b $**.

---

### Step 1: Let $ s = a + b $ and $ p = ab $

Given:
$$
ab = a + b + 3 \Rightarrow p = s + 3
$$

Our goal is to find the range of $ s = a + b $, given that $ a, b > 0 $ and $ ab = a + b + 3 $.

So we have:
$$
ab = s + 3
$$

Also, from algebra, we know that for any two numbers $ a, b $, the following holds:
$$
(a - b)^2 \geq 0 \Rightarrow a^2 + b^2 \geq 2ab
$$

But let's try a different approach.

---

### Step 2: Express $ ab $ in terms of $ s $

From the equation:
$$
ab = s + 3
$$

Also, recall the identity:
$$
(a + b)^2 = a^2 + 2ab + b^2
$$

But we can also express $ ab $ in terms of $ s $, and use the fact that $ a, b > 0 $, so we can treat them as roots of a quadratic equation.

---

### Step 3: Treat $ a, b $ as roots of a quadratic

Suppose $ a, b > 0 $ are roots of the quadratic equation:
$$
x^2 - sx + p = 0
$$
where 

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
model_name = "unsloth/qwen3-14b-unsloth-bnb-4bit"
adapter_path = ".//SFT/checkpoint-750/"
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
mode = PeftModel.from_pretrained(model, adapter_path)


# prepare the model input
prompt = "Let a, b be positive real numbers such that ab = a + b + 3. Determine the range of possible values for a + b."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

thinking content: 
content: To find the range of possible values for $a + b$, we start by using the given equation:

$$ ab = a + b + 3 $$

Our goal is to express $a + b$ in terms of a single variable and then find the range of that expression.

Let's solve the equation for one of the variables, say $a$:

$$ ab - a = b + 3 $$
$$ a(b - 1) = b + 3 $$
$$ a = \frac{b + 3}{b - 1} $$

Now, let's express $a + b$ in terms of $b$:

$$ a + b = \frac{b + 3}{b - 1} + b $$

We can simplify this expression:

$$ a + b = \frac{b + 3 + b(b - 1)}{b - 1} $$
$$ a + b = \frac{b + 3 + b^2 - b}{b - 1} $$
$$ a + b = \frac{b^2 + 3}{b - 1} $$

Now, we need to find the range of this expression for $b > 1$ (since $b$ must be positive and $b - 1$ cannot be zero). Let's analyze the behavior of the function $f(b) = \frac{b^2 + 3}{b - 1}$ for $b > 1$:

1. As $b \to 1^+$, $f(b) \to \infty$ (since the denominator approaches zero and the numerator approaches a positive constant).
2. As $b \to \infty$, $f(b) \to \infty$ (

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
model_name = "unsloth/qwen3-14b-unsloth-bnb-4bit"
adapter_path = "./SFT/checkpoint-750/"
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
mode = PeftModel.from_pretrained(model, adapter_path)


# prepare the model input
prompt = "Let a, b be positive real numbers such that ab = a + b + 3. Determine the range of possible values for a + b."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

thinking content: 
content: We are given that a and b are positive real numbers such that ab = a + b + 3. We want to find the range of possible values for a + b. Let's solve for b in terms of a and plug it into the expression for a + b.

From the given equation, we have:

ab = a + b + 3

=> b(a - 1) = a + 3

=> b = (a + 3)/(a - 1)

Now, let's find the expression for a + b in terms of a:

a + b = a + (a + 3)/(a - 1)

To find the range of possible values for a + b, we need to find the minimum and maximum values of the function f(a) = a + (a + 3)/(a - 1) for a > 1 (since a and b are positive real numbers and b = (a + 3)/(a - 1) must also be positive).

To find the minimum value of f(a), we can take the derivative of f(a) with respect to a and set it equal to zero:

f'(a) = 1 - (a + 3)/(a - 1)^2

Setting f'(a) = 0:

1 - (a + 3)/(a - 1)^2 = 0

=> (a + 3)/(a - 1)^2 = 1

=> (a + 3) = (a - 1)^2

=> a^2 - 2a + 1 = a + 3

=> a^2 - 3a - 2 = 0

Solving this quadratic equation, we get:

a = [3 ± sq

In [ ]:
%env VLLM_USE_V1=0
from vllm.vllm import LLM, SamplingParams
from vllm.vllm.lora.request import LoRARequest

lora_path="./SFT-emb/checkpoint-750/"

# Sample prompts.
prompts = [[{"role": "user", "content": ""}]]

# Create a sampling params object.
sampling_params = SamplingParams(temperature=0.6, top_k=20, top_p=0.95, max_tokens=8192)

# Create an LLM.
llm = LLM(model="unsloth/qwen3-14b-unsloth-bnb-4bit", quantization="bitsandbytes", load_format="bitsandbytes", enable_lora=True, max_lora_rank=32)
# Generate texts from the prompts. The output is a list of RequestOutput objects
# that contain the prompt, generated text, and other information.
outputs = llm.chat(prompts, sampling_params, lora_request=LoRARequest("adapter", 1, lora_path), chat_template_kwargs={"enable_thinking": False})
# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

outputs = llm.chat(prompts, sampling_params, lora_request=LoRARequest("adapter", 1, lora_path), chat_template_kwargs={"enable_thinking": True})
# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

env: VLLM_USE_V1=0
INFO 05-05 00:34:21 [__init__.py:239] Automatically detected platform cuda.


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

INFO 05-05 00:34:44 [config.py:752] This model supports multiple tasks: {'generate', 'classify', 'reward', 'embed', 'score'}. Defaulting to 'generate'.
WARNING 05-05 00:34:45 [config.py:866] bitsandbytes quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 05-05 00:34:45 [arg_utils.py:1401] The model has a long context length (40960). This may causeOOM during the initial memory profiling phase, or result in low performance due to small KV cache size. Consider setting --max-model-len to a smaller value.
INFO 05-05 00:34:45 [llm_engine.py:240] Initializing a V0 LLM engine (v0.8.5.dev456+g2858830c3) with config: model='unsloth/qwen3-14b-unsloth-bnb-4bit', speculative_config=None, tokenizer='unsloth/qwen3-14b-unsloth-bnb-4bit', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=LoadFor

tokenizer_config.json:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 05-05 00:37:13 [model_runner.py:1661] Graph capturing finished in 53 secs, took 1.16 GiB
INFO 05-05 00:37:13 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 67.53 seconds
INFO 05-05 00:37:19 [chat_utils.py:397] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

ValueError: vLLM only supports modules_to_save being None.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
model_name = "unsloth/qwen3-14b-unsloth-bnb-4bit"
adapter_path = "./SFT-emb/checkpoint-750/"
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
mode = PeftModel.from_pretrained(model, adapter_path)


# prepare the model input
prompt = "Let a, b be positive real numbers such that ab = a + b + 3. Determine the range of possible values for a + b."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


tokenizer_config.json:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.59G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

thinking content: <think>

</think>
content: We are given that ab = a + b + 3, where a and b are positive real numbers. We need to find the range of possible values for a + b. Let's denote a + b as x. Then, we can express ab in terms of x:

ab = a + b + 3
ab = x + 3

We also know that a + b = x, so we can express a in terms of b and x:

a = x - b

Now, let's substitute a in terms of b and x into the equation for ab:

(x - b) * b = x + 3
xb - b^2 = x + 3

Now, we can rearrange the equation:

b^2 - xb + x + 3 = 0

This is a quadratic equation in terms of b. For the equation to have real solutions for b, the discriminant must be non-negative:

(-x)^2 - 4 * 1 * (x + 3) ≥ 0
x^2 - 4x - 12 ≥ 0

Now, let's solve the inequality:

x^2 - 4x - 12 ≥ 0

We can factor the quadratic expression:

(x - 6)(x + 2) ≥ 0

To find the range of x, we can examine the intervals defined by the factors:

1. When x ≤ -2, both factors are negative, so their product is positive.
2. When -2 ≤ x ≤ 6, one factor is nega

# All packages and versions installed (for reproducibility)

In [ ]:
!pip freeze

absl-py==1.4.0
accelerate==1.6.0
aiohappyeyeballs==2.6.1
aiohttp==3.11.15
aiosignal==1.3.2
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.6
ale-py==0.11.0
altair==5.5.0
annotated-types==0.7.0
anyio==4.9.0
argon2-cffi==23.1.0
argon2-cffi-bindings==21.2.0
array_record==0.7.2
arviz==0.21.0
astropy==7.0.1
astropy-iers-data==0.2025.4.28.0.37.27
astunparse==1.6.3
atpublic==5.1
attrs==25.3.0
audioread==3.0.1
auto_gptq==0.7.1
autograd==1.7.0
babel==2.17.0
backcall==0.2.0
backports.tarfile==1.2.0
beautifulsoup4==4.13.4
betterproto==2.0.0b6
bigframes==2.1.0
bigquery-magics==0.9.0
bitsandbytes==0.45.5
bleach==6.2.0
blinker==1.9.0
blis==1.3.0
blosc2==3.3.1
bokeh==3.7.2
Bottleneck==1.4.2
bqplot==0.12.44
branca==0.8.1
build==1.2.2.post1
CacheControl==0.14.2
cachetools==5.5.2
catalogue==2.0.10
certifi==2025.4.26
cffi==1.17.1
chardet==5.2.0
charset-normalizer==3.4.1
chex==0.1.89
clarabel==0.10.0
click==8.1.8
cloudpathlib==0.21.0
cloudpickle==3.1.1
cmake==3.31.6
cmdstanpy==1.2.5
colorcet==3.1.0
